### Loading Libraries

In [69]:
import os
from dotenv import load_dotenv
import praw
from openai import OpenAI
import pandas as pd
import time
from pinecone import Pinecone
import json

### Connect APIs

In [52]:
### Setup
load_dotenv('../.env.local')
REDDIT_CLIENT_ID = os.environ.get('REDDIT_CLIENT_ID')
REDDIT_SECRET_ID = os.environ.get('REDDIT_SECRET_ID')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')


In [144]:
### Reddit API
reddit = praw.Reddit(
    client_id = REDDIT_CLIENT_ID,
    client_secret = REDDIT_SECRET_ID,
    user_agent = "researcher"
)
# Session options: controversial, gilded, hot, new, rising, top
print(reddit.read_only)

### OpenAI API
client = OpenAI(api_key=OPENAI_API_KEY)

### Pinecone API
pc = Pinecone(api_key=PINECONE_API_KEY)

# Create a new index (or connect to existing one)
index_name = "flight-reviews"
dimension = 1536  # text-embedding-3-small produces 1536-dimensional vectors

# Check if index exists, create if it doesn't
existing_indexes = [idx.name for idx in pc.list_indexes()]
if index_name not in existing_indexes:
    print(f"Creating new index: {index_name}")
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",  # cosine similarity is standard for embeddings
        spec={
            "serverless": {
                "cloud": "aws",
                "region": "us-east-1"  # Change to your preferred region
            }
        }
    )

index = pc.Index(index_name)
print(f"Connected to index: {index_name}")

True
Creating new index: flight-reviews
Connected to index: flight-reviews


In [142]:

pc.delete_index(name="flight-reviews")


In [145]:
indexes = [idx.name for idx in pc.list_indexes()]
print(indexes)

['tutor-index', 'reddit', 'demo', 'flight-reviews']


### Scrape Reddit

In [171]:
### get comments and posts
target_subreddits = reddit.subreddit("delta+united+AmericanAirlines+travel+awardtravel")
search_query = '''
                ("Business Class" OR "First Class" OR "D1" OR "Polaris") 
                AND 
                ("App" OR "Mobile Boarding" OR "Online Check-in" OR "App Crash" OR "System Error" OR "Website Glitch" OR "Boarding Pass won't load")
               '''

title, text, score, url = [], [], [], []
for submission in target_subreddits.search(
    search_query,
    sort="relevance",
    time_filter="year",
    limit=500):

    title.append(submission.title)
    text.append(submission.selftext)
    score.append(submission.score)
    url.append(submission.url)
    print(f'Unique Submissions {len(set(title))}', end='\r', flush=True)
    time.sleep(0.01)

In [172]:
reddit_df = pd.DataFrame({
    'Title': title,
    'Text': text,
    'Score': score,
    'URL': url
})
display(reddit_df)

,Title,Text,Score,URL
0,Downgraded to economy on a paid first class ti...,"I’m 6’5” and a pretty big dude, so I usually b...",3272,https://i.redd.it/vps8xlmkbo7e1.jpeg
1,Frustrated about Delta employees and upgrades ...,Happened today - going to hide the airport pai...,673,https://www.reddit.com/r/delta/comments/1nox4g...
2,Just WOW,Nothing like pulling up to the airport and fin...,1004,https://i.redd.it/zlqfszmix68e1.jpeg
3,"Is the price to ""upgrade"" to Delta One always ...",I rarely buy a first class ticket because they...,0,https://www.reddit.com/r/delta/comments/1j30eq...
4,Delta -- The worst deal in the skies?,\n\nDiamond Medallion for 7 years straight her...,424,https://www.reddit.com/r/delta/comments/1o50fi...
...,...,...,...,...
236,What can I reasonably expect? (See caption),I booked this flight a couple months ago and n...,2,https://www.reddit.com/gallery/1khnkph
237,A little love for comfort+,Just came back from Zurich on an old ass 767. ...,22,https://www.reddit.com/r/delta/comments/1ly9zo...
238,Upgrade to First Class?,I booked a work trip to Detroit through the wo...,5,https://www.reddit.com/r/americanairlines/comm...
239,Chances of upgrade prices going up/down based ...,Hi! I have two C+ seats booked on a flight fro...,0,https://www.reddit.com/r/delta/comments/1ls3gc...


### Vectorize Posts

In [173]:
# Combine title and text then filter out empty/removed Posts
posts = [f'{title[i]} --> {text[i]}' for i in range(len(title))]
print(f"Total posts to embed: {len(posts):,}")

Total posts to embed: 241


##### Batch Embeddings

In [174]:
# Set batch and comment char length
batch_size = 200
max_chars = 200000

all_vectors = []
total_batches = (len(posts) + batch_size - 1) // batch_size
print(f'Processing {total_batches} batches of up to {batch_size} posts')

for batch_num in range(0, len(posts), batch_size):
    batch = posts[batch_num:batch_num + batch_size]
    batch_char = sum(len(c) for c in batch)

    print(f'Batch {(batch_num // batch_size) + 1}/{total_batches}: Embedding {len(batch)} posts {batch_char:,} chars...', end=" ")

    embeddings_response = client.embeddings.create(
        model="text-embedding-3-small",
        input=batch
    )

    vectors = [
        (f'id_{batch_num}_{j}', embeddings_response.data[j].embedding, {'text': batch[j], 'upvotes': score[batch_num + j], 'url': url[batch_num + j]})
        for j in range(len(batch))
    ]

    all_vectors.extend(vectors)
    print(f'Done ({len(all_vectors):,} vectors total)')

Processing 2 batches of up to 200 posts
Batch 1/2: Embedding 200 posts 259,437 chars... Done (200 vectors total)
Batch 2/2: Embedding 41 posts 49,639 chars... Done (241 vectors total)


##### Upsert to Pinecone

In [175]:
upsert_batch_size = 100
for i in range(0, len(all_vectors), upsert_batch_size):
    batch = all_vectors[i:i+upsert_batch_size]
    index.upsert(vectors=batch)
    print(f'Uploading {i + len(batch)}/{len(all_vectors)} vectors')
print(f'Successfully uploaded {len(all_vectors)} vectors to Pinecone')

Uploading 100/241 vectors
Uploading 200/241 vectors
Uploading 241/241 vectors
Successfully uploaded 241 vectors to Pinecone


### Query Relevant Comments

In [176]:
query = "The airline mobile app crashed and I couldn't access my boarding pass for business class."
query_embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input=query
    ).data[0].embedding

results = index.query(vector=query_embedding, top_k=10, include_metadata=True)

for match in results['matches']:
    print(f'Score: {match['score']:.3f}')
    print(f'Post {match['metadata']['text']}\n')

Score: 0.506
Post When premium economy isn't premium --and a question about upgrading --> We took an AA flight to Barcelona and had 4 seats in Premium Economy,. I certainly expected a better experience considering what I spent

The issues:

1. There were two lines to board the flight. One for groups 1-2 (Business class and Premium Economy) and another for everyone else. But everyone boarded at the same time, so it was a crush of people --what is the purpose of separating these groups just to announce that everyone can board all at the same time?

2. My wife was called back and told that her entertainment system was broken on the plane. My entrainment system was also broken, so no movies.

3. It was an overnight flight, and after the flight attendant served dinner and a drink one hour in, she vanished until 6 hours later. Lights out, no drinks. I can't sleep on the plane, so I wanted to sip wine and read. Couldn't. 

(the food was quite good and the flight was on-time, so that was a pos

In [184]:
# Save results in Markdown format for better readability
markdown_content = f"""# Reddit Query Results

## Query
{query}

## Results (Top {len(results['matches'])} matches)

"""
for i, match in enumerate(results['matches'], 1):
    score = match['score']
    text = match['metadata']['text']
    markdown_content += f""" ### Result {i} (Similarity Score: {score:.3f}) [Link]({match['metadata']['url']})

{text}

---

"""

with open('../datasets/reddit/top_comments.md', 'w', encoding='utf-8') as file:
    file.write(markdown_content)

print(f"Results saved to top_comments.md")

# Optionally, also save as JSON for programmatic access
results_dict = {}
for match in results['matches']:
    results_dict[match['score']] = match['metadata']['text']

with open('../datasets/reddit/top_comments.json', 'w', encoding='utf-8') as file:
    json.dump(results_dict, file, indent=4, ensure_ascii=False)

print(f"Results also saved to top_comments.json for programmatic access")

Results saved to top_comments.md
Results also saved to top_comments.json for programmatic access
